In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import xgboost as xgb
import joblib
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [78]:
# Load the dataset
data = pd.read_csv("C:/Users/ganmi/Downloads/property_insurance_historical_data.csv")

print(f"Dataset shape: {data.shape[0]} rows, {data.shape[1]} columns")

# Print first few rows
print("\n First 5 rows:")
print(data.head())

# Print summary statistics (numeric columns only)
print("\n Summary statistics:")
print(data.describe())

# Print column names and data types
print("\n Column types:")
print(data.dtypes)

Dataset shape: 63189 rows, 38 columns

 First 5 rows:
      earned_exposure       policy_id underwriting_year policy_type  \
0  0.9252155845351104  TRNPOL000000R1              2021  homeownner   
1  0.9819822278201685  TRNPOL000001R2              2022  homeownner   
2  0.8151977797193338  TRNPOL000002R3              2019  homeownner   
3  0.9808944570661171  TRNPOL000003R4              2020  homeownner   
4  0.9177086469447747  TRNPOL000004R5              2023  homeownner   

  coverage_limit policy_duration recent_claim_activity underwriting_score  \
0       443800.0               4                     Y               92.0   
1       134400.0               2                     Y               79.0   
2       115000.0               3                     Y               99.0   
3       314100.0               1                     Y               94.0   
4       486700.0               5                     Y               87.0   

  has_deductible property_type  ...  is_lightning_prone_

C:\Users\ganmi\AppData\Local\Temp\ipykernel_31356\2430044698.py:2: DtypeWarning: Columns (0,2,4,5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("C:/Users/ganmi/Downloads/property_insurance_historical_data.csv")


In [79]:
# Handle Missing Value
bad_indices = ["-1", 9999, -20, "polystyrene"]  # remove the bad rows
data = data[~data.isin(bad_indices).any(axis=1)]

# Replace 999999 in 'prev_inspect_ago' with 0
data.loc[data['prev_inspect_ago'] == 999999, 'prev_inspect_ago'] = 0

# Where 'has_inspect' is 'N', set 'prev_inspect_ago' to 0
data.loc[data['has_inspect'].astype(str).str.upper().str.strip() == 'N', 'prev_inspect_ago'] = 0 # Replace 999999 in 'prev_inspect_ago' with 0
data.loc[data['prev_inspect_ago'] == 999999, 'prev_inspect_ago'] =0

# Where 'has_inspect' is 'N', set 'prev_inspect_ago' to 0
data.loc[data['has_inspect'].astype(str).str.upper().str.strip() == 'N', 'prev_inspect_ago'] = 0

In [80]:
# Define target variable
y = data['total_loss_value']

# Define features (remove non-predictive columns)
X = data.drop(columns=['total_loss_value', 'theft_loss_value', 'policy_id'])

In [81]:
# Step 1: Define binary Y/N columns
binary_yn_cols = [
    'recent_claim_activity',
    'has_deductible',
    'has_sprinkler',
    'has_fire_hydrant',
    'has_fire_extinguishers',
    'has_smoke_alarm',
    'has_inspect',
    'last_inspection_result',
    'has_guard_house',
    'has_hazardeous_items',
    'has_attended_ge_risk_prevention_course',
    'has_fire_dept_nearby',
    'local_fire_incident_rate',
    'is_lightning_prone_area',
    'high_density_area',
    'with_mortgage'
]

# Convert Y/N columns to binary 1/0
for col in binary_yn_cols:
    if col in data.columns:
        data[col] = data[col].astype(str).str.upper().str.strip().map({'Y': 1, 'N': 0})

In [82]:
# Step 2: Identify categorical and numeric columns (excluding binary)
categorical_cols = ["incident_time", "locations", "policy_type", "property_type", "state_code","underwriting_year"]

numeric_cols = [ "building_age", "coverage_limit", "distance_to_fire_dept_km", "earned_exposure","num_floors", 
                "policy_duration", "power_usage", "prev_inspect_ago","underwriting_score"]

In [83]:
# Force all categorical columns to string type
for col in categorical_cols:
    data[col] = data[col].astype(str)

# One-hot encode categorical columns
if categorical_cols:
    encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
    X_categorical = encoder.fit_transform(data[categorical_cols])
    print(f"✅ One-hot encoded categorical shape: {X_categorical.shape}")
else:
    X_categorical = None

# Min-max normalize numeric columns
if numeric_cols:
    scaler = MinMaxScaler()
    X_numeric = scaler.fit_transform(data[numeric_cols])
    print(f"✅ Min-max normalized numeric shape: {X_numeric.shape}")
else:
    X_numeric = None

# Extract binary columns as is
if binary_yn_cols:
    X_binary = data[binary_yn_cols].values
    print(f"✅ Binary columns shape: {X_binary.shape}")
else:
    X_binary = None

# Combine all parts
to_combine = [arr for arr in [X_numeric, X_binary, X_categorical] if arr is not None]
if to_combine:
    X_processed = np.hstack(to_combine)
    print(f"✅ Final combined feature matrix shape: {X_processed.shape}")
else:
    raise ValueError("No data left after processing!")
    
df_preprocessed = pd.DataFrame(X_processed)
df_preprocessed.to_csv("preprocessed.csv", index=False)

✅ One-hot encoded categorical shape: (63185, 23)
✅ Min-max normalized numeric shape: (63185, 9)
✅ Binary columns shape: (63185, 16)
✅ Final combined feature matrix shape: (63185, 48)


In [42]:
# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=123)

# Convert your train/test arrays to DMatrix format (required for xgb.train)
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test, label=y_test)

### Model Training

In [34]:
!pip install optuna
!pip install xgboost
!pip install --upgrade xgboost

In [51]:
def objective(trial):
    params = {
        "objective": "reg:tweedie",
        "tweedie_variance_power": trial.suggest_float("tweedie_p", 1.2, 1.7),
        "learning_rate": trial.suggest_float("eta", 0.005, 0.15),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
        "subsample": trial.suggest_float("subsample", 0.5, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),
        "gamma": trial.suggest_float("gamma", 0, 6),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
        "reg_alpha":  trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
        "tree_method": "hist",
        "seed": 123
    }

    booster = xgb.train(
        params,
        dtrain,
        num_boost_round=3000,
        evals=[(dtest, "valid")],
        early_stopping_rounds=150,
        verbose_eval=False
    )

    preds = booster.predict(dtest)       # already RM scale with Tweedie
    rmse  = mean_squared_error(y_test, preds, squared=False)
    trial.set_user_attr("best_iteration", booster.best_iteration)
    return rmse                          # Optuna minimises


In [52]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=100, show_progress_bar=True)

best_params = study.best_params
best_params.update({
    "tree_method": "hist",
    "objective": "reg:squarederror",
    "random_state": 123,
    "n_estimators": study.best_trial.user_attrs.get("best_iteration", 1000)
})

print("\n✅ Best hyperparameters:")
for k, v in best_params.items():
    print(f"{k}: {v}")

[I 2025-05-08 19:22:41,115] A new study created in memory with name: no-name-bae37797-d2f8-4992-8d93-d84cb7aaced7


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2025-05-08 19:22:41,856] Trial 0 finished with value: 695394.4287795043 and parameters: {'tweedie_p': 1.3909786430260775, 'eta': 0.10941287077877959, 'max_depth': 5, 'min_child_weight': 4, 'subsample': 0.8061281980502593, 'colsample_bytree': 0.5045540222626363, 'gamma': 2.2241021335687376, 'reg_lambda': 0.09881638373172084, 'reg_alpha': 0.09384296505597882}. Best is trial 0 with value: 695394.4287795043.
[I 2025-05-08 19:22:43,843] Trial 1 finished with value: 720415.107758166 and parameters: {'tweedie_p': 1.4841849126616151, 'eta': 0.12520247497630504, 'max_depth': 10, 'min_child_weight': 5, 'subsample': 0.8614908835351094, 'colsample_bytree': 0.7297349435070547, 'gamma': 5.354102593605525, 'reg_lambda': 0.11129067415459193, 'reg_alpha': 0.00012313518055263281}. Best is trial 0 with value: 695394.4287795043.
[I 2025-05-08 19:22:44,789] Trial 2 finished with value: 687377.1355826951 and parameters: {'tweedie_p': 1.4855917494003976, 'eta': 0.042062013384684314, 'max_depth': 3, 'min_c

In [53]:
# ---------------------------------------------------------
# 3. Retrain with best params
# ---------------------------------------------------------
best_iter = study.best_trial.user_attrs.get("best_iteration", 1000)
best_params.update({
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "seed": 123
})

final_model = xgb.train(
    best_params,
    dtrain,
    num_boost_round=best_iter,
    evals=[(dtrain, "train")],
    verbose_eval=False
)

C:\Users\ganmi\anaconda3\lib\site-packages\xgboost\core.py:158: UserWarning: [19:27:29] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "n_estimators", "tweedie_p" } are not used.

  warnings.warn(smsg, UserWarning)


In [54]:
# ---------------------------------------------------------
# 4. Predict & Evaluate
# ---------------------------------------------------------
preds = final_model.predict(dtest)
rmse = mean_squared_error(y_test, preds, squared=False)
mae  = mean_absolute_error(y_test, preds)

print("\n📊 Final Model Performance:")
print(f"RMSE: {rmse:,.2f}")
print(f"MAE : {mae:,.2f}")




📊 Final Model Performance:
RMSE: 685,860.95
MAE : 213,137.47


In [65]:
# Load the dataset
data = pd.read_csv("C:/Users/ganmi/Downloads/final.csv")

print(f"Dataset shape: {data.shape[0]} rows, {data.shape[1]} columns")

# Print first few rows
print("\n First 5 rows:")
print(data.head())

# Print summary statistics (numeric columns only)
print("\n Summary statistics:")
print(data.describe())

# Print column names and data types
print("\n Column types:")
print(data.dtypes)

Dataset shape: 13298 rows, 36 columns

 First 5 rows:
   earned_exposure       policy_id  underwriting_year policy_type  \
0         0.917073  TSTPOL000000R1               2019  industrial   
1         0.627704  TSTPOL000001R2               2019  industrial   
2         0.975467  TSTPOL000002R3               2023  homeownner   
3         0.950129  TSTPOL000003R4               2020  homeownner   
4         0.821225  TSTPOL000004R5               2020  homeownner   

   coverage_limit  policy_duration recent_claim_activity  underwriting_score  \
0        13544600                1                     Y                  94   
1        10061100                4                     N                  78   
2          125500                0                     Y                  86   
3          158700                2                     N                  89   
4          205600                1                     Y                  78   

  has_deductible property_type  ...  state_code  l

In [71]:
# Handle Missing Value
bad_indices = ["-1", 9999, -20, "polystyrene"]  # remove the bad rows
data = data[~data.isin(bad_indices).any(axis=1)]

# Replace 999999 in 'prev_inspect_ago' with 0
data.loc[data['prev_inspect_ago'] == 999999, 'prev_inspect_ago'] = 0

# Where 'has_inspect' is 'N', set 'prev_inspect_ago' to 0
data.loc[data['has_inspect'].astype(str).str.upper().str.strip() == 'N', 'prev_inspect_ago'] = 0 # Replace 999999 in 'prev_inspect_ago' with 0
data.loc[data['prev_inspect_ago'] == 999999, 'prev_inspect_ago'] =0

# Where 'has_inspect' is 'N', set 'prev_inspect_ago' to 0
data.loc[data['has_inspect'].astype(str).str.upper().str.strip() == 'N', 'prev_inspect_ago'] = 0

In [73]:
# Step 1: Define binary Y/N columns
binary_yn_cols = [
    'recent_claim_activity',
    'has_deductible',
    'has_sprinkler',
    'has_fire_hydrant',
    'has_fire_extinguishers',
    'has_smoke_alarm',
    'has_inspect',
    'last_inspection_result',
    'has_guard_house',
    'has_hazardeous_items',
    'has_attended_ge_risk_prevention_course',
    'has_fire_dept_nearby',
    'local_fire_incident_rate',
    'is_lightning_prone_area',
    'high_density_area',
    'with_mortgage'
]

# Convert Y/N columns to binary 1/0
for col in binary_yn_cols:
    if col in data.columns:
        data[col] = data[col].astype(str).str.upper().str.strip().map({'Y': 1, 'N': 0})

In [74]:
# Step 2: Identify categorical and numeric columns (excluding binary)
categorical_cols = ["incident_time", "locations", "policy_type", "property_type", "state_code","underwriting_year"]

numeric_cols = [ "building_age", "coverage_limit", "distance_to_fire_dept_km", "earned_exposure","num_floors", 
                "policy_duration", "power_usage", "prev_inspect_ago","underwriting_score"]

In [75]:
# Force all categorical columns to string type
for col in categorical_cols:
    data[col] = data[col].astype(str)

# One-hot encode categorical columns
if categorical_cols:
    encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
    X_categorical = encoder.fit_transform(data[categorical_cols])
    print(f"✅ One-hot encoded categorical shape: {X_categorical.shape}")
else:
    X_categorical = None

# Min-max normalize numeric columns
if numeric_cols:
    scaler = MinMaxScaler()
    X_numeric = scaler.fit_transform(data[numeric_cols])
    print(f"✅ Min-max normalized numeric shape: {X_numeric.shape}")
else:
    X_numeric = None

# Extract binary columns as is
if binary_yn_cols:
    X_binary = data[binary_yn_cols].values
    print(f"✅ Binary columns shape: {X_binary.shape}")
else:
    X_binary = None

# Combine all parts
to_combine = [arr for arr in [X_numeric, X_binary, X_categorical] if arr is not None]
if to_combine:
    X_processed = np.hstack(to_combine)
    print(f"✅ Final combined feature matrix shape: {X_processed.shape}")
else:
    raise ValueError("No data left after processing!")

✅ One-hot encoded categorical shape: (13298, 23)
✅ Min-max normalized numeric shape: (13298, 9)
✅ Binary columns shape: (13298, 16)
✅ Final combined feature matrix shape: (13298, 48)


In [89]:
# ---------------------------------------------------------------------------
# 5.  Predict total_loss_value for the new policies in final.csv
# ---------------------------------------------------------------------------
FINAL_PATH  = "C:/Users/ganmi/Downloads/final.csv"
OUT_PATH    = "C:/Users/ganmi/Downloads/final_with_predictions.csv"

final_df = pd.read_csv(FINAL_PATH)

# ---------- 5A.  Fast clean identical to training --------------------------
# remove “bad” rows
bad_vals = {"-1", 9999, -20, "polystyrene"}
final_df = final_df[~final_df.isin(bad_vals).any(axis=1)]

# specific replacements
final_df.loc[final_df["prev_inspect_ago"] == 999999, "prev_inspect_ago"] = 0
final_df["has_inspect"] = final_df["has_inspect"].astype(str).str.upper().str.strip()
final_df.loc[final_df["has_inspect"] == "N", "prev_inspect_ago"] = 0

# ------------ 5B.  Binary Y/N remap ----------------------------------------
for col in binary_yn_cols:
    if col in final_df.columns:
        final_df[col] = (
            final_df[col].astype(str).str.upper().str.strip().map({"Y": 1, "N": 0})
        )
    else:                                  # column absent in new file → assume 0
        final_df[col] = 0

# ------------ 5C.  One-hot + min-max using *trained* encoder/scaler ---------
# force cat to str
for col in categorical_cols:
    if col in final_df.columns:
        final_df[col] = final_df[col].astype(str)
    else:                                  # unseen col in new file → fill "nan"
        final_df[col] = "nan"

# Numeric
X_num_final = scaler.transform(final_df[numeric_cols])           # min-max
# Binary
X_bin_final = final_df[binary_yn_cols].values
# Categorical
X_cat_final = encoder.transform(final_df[categorical_cols])      # one-hot (ignore unknown)

# stack in the same order as training
X_final = np.hstack([X_num_final, X_bin_final, X_cat_final])

# ------------- 5D.  Predict with the trained booster -----------------------
dfinal = xgb.DMatrix(X_final)
final_df["predicted_total_loss_value"] = final_model.predict(dfinal)
load_factor = 1 - 0.1987          # 0.8013

# ----------- NEW:  premium = loss / (1 – 0.0787 – 0.05 – 0.07) -------------
load_factor = 1 - (0.0787 + 0.05 + 0.07)    # 0.8013
final_df["premium"] = (final_df["predicted_total_loss_value"] / load_factor).clip(lower=0)

# ------------- 5E.  Save & preview -----------------------------------------
final_df.to_csv(OUT_PATH, index=False)
print("✅ Predictions written to:", OUT_PATH)
print("\n🔍 Preview:")
print(final_df[["policy_id", "predicted_total_loss_value", "premium"]].head())

# ------------- 5E.  Save & preview -----------------------------------------
final_df.to_csv(OUT_PATH, index=False)
print("✅ Predictions written to:", OUT_PATH)
print("\n🔍 Preview:")
print(final_df[["policy_id", "predicted_total_loss_value"]].head())


✅ Predictions written to: C:/Users/ganmi/Downloads/final_with_predictions.csv

🔍 Preview:
        policy_id  predicted_total_loss_value        premium
0  TSTPOL000000R1                -3873.501709       0.000000
1  TSTPOL000001R2               426581.562500  532361.875000
2  TSTPOL000002R3                 1460.035645    1822.083740
3  TSTPOL000003R4                -3864.048828       0.000000
4  TSTPOL000004R5                28106.429688   35076.039062
✅ Predictions written to: C:/Users/ganmi/Downloads/final_with_predictions.csv

🔍 Preview:
        policy_id  predicted_total_loss_value
0  TSTPOL000000R1                -3873.501709
1  TSTPOL000001R2               426581.562500
2  TSTPOL000002R3                 1460.035645
3  TSTPOL000003R4                -3864.048828
4  TSTPOL000004R5                28106.429688
